# SPLATONIC-on-MonoGS: validating the constant-angular-velocity motion prior fix

**Background** (full writeup: `port/STATUS.md` section 9 in the repo). Porting SPLATONIC's
sparse pixel-based Gaussian-splat rasterization onto MonoGS (monocular Gaussian SLAM) works
end-to-end, but sparse tracking's rotation estimate drifts badly on TUM `fr1_desk`:
RMSE ATE ~0.72m vs. dense's ~0.03m. Root-caused (two independent lines of evidence -- a live
A/B loss-switching test, and direct per-frame gradient-variance measurement at ground truth)
to: **sparse tracking's per-frame rotation gradient is a high-variance, not fixed-bias,
Monte-Carlo estimate of the true photometric gradient**, and since each frame seeds its pose
from the previous frame's estimate with no loop closure, this noise compounds into an
uncorrected random walk in rotation (translation is unaffected).

Two cheap fixes were already tried and **failed**: more pixels per tile (4x, no improvement)
and periodic dense re-anchoring (fixes instantaneous drift but not the whole-trajectory RMSE
metric, which sums every excursion along the way). This notebook validates the third,
untried direction: a **constant-angular-velocity motion prior** that fuses each frame's noisy
rotation estimate with a lower-variance constant-velocity prediction from the previous frame
(shrinkage estimator, `tracking_motion_prior_alpha` in `slam_frontend.py`).

**Runs (5 total, TUM `fr1_desk`, ~613 frames each):**
1. `fr1_desk.yaml` -- dense baseline (sanity check, expect RMSE ATE ~0.03m)
2. `fr1_desk_splatonic.yaml` -- sparse baseline (the regression, expect ~0.72m)
3. `fr1_desk_splatonic_motionprior_030.yaml` -- sparse + motion prior, alpha=0.3
4. `fr1_desk_splatonic_motionprior.yaml` -- sparse + motion prior, alpha=0.5
5. `fr1_desk_splatonic_motionprior_075.yaml` -- sparse + motion prior, alpha=0.75

Expected total time on a T4: on the dev machine's 4GB GPU each run processed frames at
~0.5-0.8 FPS end-to-end (~15-25 min/run); a T4 should be comparable or faster. Budget
~2-3 hours total, well inside a 30-hour quota -- there's room to re-run or sweep more alpha
values if the first pass is inconclusive.

## 0. Before running: push your latest local commits

This notebook clones `https://github.com/kislay536/temp.git` (public). Make sure the local
repo's latest commits -- including the motion-prior fix and the headless-import fix in
`slam.py` -- are pushed to `origin/main` first, otherwise Kaggle will pull stale code.
If you'd rather not push, upload the repo as a Kaggle Dataset instead and change the clone
cell below to copy from `/kaggle/input/<dataset-name>/` instead of `git clone`.

## 1. Environment check

In [ ]:
import subprocess, torch, sys
print(sys.version)
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout)

## 2. Clone the repo

In [ ]:
%cd /kaggle/working
!rm -rf temp
!git clone --depth 1 https://github.com/kislay536/temp.git
%cd temp/MonoGS
!git log --oneline -3

## 3. System packages (best-effort)

`opencv-python` and `open3d` need a few shared libraries even in headless/import-only mode.
These installs are non-fatal if `apt-get` isn't available or some packages are already present.

In [ ]:
!apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 || true

## 4. Python dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q opencv-python munch trimesh 'evo==1.11.0' open3d torchmetrics rich plyfile wandb lpips
# glfw / PyOpenGL / imgviz are only needed for the interactive GUI (use_gui=True), which
# these headless --eval runs never touch (slam.py now defers that import -- see the
# 'defer GUI-only imports' commit). Skipping them removes a real source of headless-install pain.

## 5. Build the CUDA extensions

`simple-knn` isn't vendored in this checkout (only `diff-gaussian-rasterization`,
`track-rasterization`, `map-rasterization` are) -- fetch it from its upstream repo, matching
`.gitmodules`. `TORCH_CUDA_ARCH_LIST=7.5` targets the T4 explicitly and speeds up the build.

In [ ]:
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
!rm -rf submodules/simple-knn
!git clone --depth 1 https://gitlab.inria.fr/bkerbl/simple-knn.git submodules/simple-knn

In [ ]:
import subprocess

build_logs = {}
build_failed = []
os.makedirs('/kaggle/working/build_logs', exist_ok=True)
for pkg_dir in [
    'submodules/simple-knn',
    'submodules/diff-gaussian-rasterization',
    'track-rasterization',
    'map-rasterization',
]:
    print('=== building', pkg_dir, '===')
    log_path = f"/kaggle/working/build_logs/{pkg_dir.replace('/', '_')}.log"
    build_logs[pkg_dir] = log_path
    # No -q here on purpose: -q previously swallowed the actual nvcc/gcc error and
    # only surfaced pip's generic 'did not run successfully' wrapper. Full output goes
    # to a log file (can be long) and the tail prints inline on failure.
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['pip', 'install', '-e', '.', '--no-build-isolation'],
            cwd=pkg_dir, stdout=logf, stderr=subprocess.STDOUT,
        )
    if proc.returncode != 0:
        build_failed.append(pkg_dir)
        print(f'!! build FAILED for {pkg_dir} (exit {proc.returncode}) -- last 80 lines of {log_path}:')
        with open(log_path) as f:
            lines = f.readlines()
        print(''.join(lines[-80:]))
    else:
        print(f'OK: {pkg_dir}')

if build_failed:
    print()
    print('FAILED:', build_failed)
    print('Full logs are in /kaggle/working/build_logs/ -- open the relevant .log file,')
    print('or paste the tail printed above, to diagnose the real compiler error.')
else:
    print('all extensions built')

In [ ]:
# Only run this once the build cell above reports no failures.
import track_rasterization, map_rasterization, diff_gaussian_rasterization, simple_knn
print('all 4 CUDA extensions import cleanly')

## 6. Download the TUM fr1_desk dataset

In [ ]:
!mkdir -p datasets/tum
%cd datasets/tum
!wget -q https://vision.in.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz
!tar -xzf rgbd_dataset_freiburg1_desk.tgz
%cd /kaggle/working/temp/MonoGS
!wc -l datasets/tum/rgbd_dataset_freiburg1_desk/rgb.txt

## 7. Run helper

Runs `slam.py --eval` with a given config, times it, and parses the final RMSE ATE
(`results/.../plot/stats_final.json`, evo's `rmse` field) and rendering PSNR/SSIM
(`results/.../psnr/*/final_result.json`) out of whatever new results directory the run created.

In [ ]:
import glob, json, os, subprocess, time

RESULTS_ROOT = 'results/tum_rgbd_dataset_freiburg1_desk'

def run_config(config_path, log_path):
    existing = set(glob.glob(os.path.join(RESULTS_ROOT, '*')))
    env = os.environ.copy()
    env['WANDB_MODE'] = 'disabled'
    start = time.time()
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['python', '-u', 'slam.py', '--config', config_path, '--eval'],
            env=env, stdout=logf, stderr=subprocess.STDOUT, timeout=3600,
        )
    elapsed = time.time() - start
    if proc.returncode != 0:
        print(f'!! {config_path} exited {proc.returncode} -- see {log_path}')
        return None
    new_dirs = set(glob.glob(os.path.join(RESULTS_ROOT, '*'))) - existing
    if not new_dirs:
        print(f'!! {config_path}: no new results dir found')
        return None
    save_dir = max(new_dirs, key=os.path.getmtime)

    result = {'config': config_path, 'save_dir': save_dir, 'elapsed_min': elapsed / 60}
    stats_path = os.path.join(save_dir, 'plot', 'stats_final.json')
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            stats = json.load(f)
        result['rmse_ate'] = stats.get('rmse')
    psnr_dirs = glob.glob(os.path.join(save_dir, 'psnr', '*', 'final_result.json'))
    if psnr_dirs:
        with open(psnr_dirs[0]) as f:
            psnr_stats = json.load(f)
        result['mean_psnr'] = psnr_stats.get('mean_psnr')
        result['mean_ssim'] = psnr_stats.get('mean_ssim')
    return result

## 8. Run all 5 configs

In [ ]:
CONFIGS = [
    ('dense_baseline', 'configs/mono/tum/fr1_desk.yaml'),
    ('sparse_baseline', 'configs/mono/tum/fr1_desk_splatonic.yaml'),
    ('sparse_prior_030', 'configs/mono/tum/fr1_desk_splatonic_motionprior_030.yaml'),
    ('sparse_prior_050', 'configs/mono/tum/fr1_desk_splatonic_motionprior.yaml'),
    ('sparse_prior_075', 'configs/mono/tum/fr1_desk_splatonic_motionprior_075.yaml'),
]

os.makedirs('/kaggle/working/logs', exist_ok=True)
results = {}
for name, cfg in CONFIGS:
    print(f'--- running {name} ({cfg}) ---')
    log_path = f'/kaggle/working/logs/{name}.log'
    r = run_config(cfg, log_path)
    results[name] = r
    print(name, '->', r)

## 9. Comparison table

In [ ]:
import pandas as pd

rows = []
for name, _ in CONFIGS:
    r = results.get(name)
    if r is None:
        rows.append({'run': name, 'rmse_ate_m': None, 'mean_psnr': None, 'mean_ssim': None, 'minutes': None})
    else:
        rows.append({
            'run': name,
            'rmse_ate_m': r.get('rmse_ate'),
            'mean_psnr': r.get('mean_psnr'),
            'mean_ssim': r.get('mean_ssim'),
            'minutes': round(r.get('elapsed_min', 0), 1),
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('/kaggle/working/motion_prior_validation_results.csv', index=False)
with open('/kaggle/working/motion_prior_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print()
print('Saved: /kaggle/working/motion_prior_validation_results.{csv,json}')
print('Full per-run logs: /kaggle/working/logs/<name>.log')

## 10. How to read this

- **Sanity check:** `dense_baseline` should land near the dev-GPU reference (~0.03m RMSE ATE,
  ~21 PSNR). If it doesn't, something about the environment differs from the dev box and the
  other numbers below aren't trustworthy either.
- **`sparse_baseline`** should reproduce the regression (~0.7m RMSE ATE, ~14 PSNR) -- this is
  the number the fix needs to beat.
- **`sparse_prior_030/050/075`:** any of these meaningfully below `sparse_baseline`'s RMSE ATE
  is a real, positive result for the motion-prior fix -- report whichever alpha wins.
  If all three land close to `sparse_baseline` (little to no improvement), the fix as designed
  doesn't address the aggregate metric (similar to why periodic dense re-anchoring failed --
  see `port/STATUS.md` section 9) and the remaining direction is proper loop closure/bundle
  adjustment, or a stronger/adaptive prior (e.g. alpha that grows with estimated variance,
  or a full temporal filter instead of two-frame linear blending).
- If a run's dict is `None` above, check `/kaggle/working/logs/<name>.log` for the actual
  traceback before concluding anything about that config.